### Imports

In [33]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import cross_validate
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

### Five Season Matches

In [2]:
season_tables = []

for season in ["2021_22", "2022_23", "2023_24", "2024_25", "2025_26"]:
    filename = f"premier_league_{season}.json"

    with open(filename, encoding="utf-8") as file:
        data = json.load(file)

    season_matches = pd.json_normalize(data["matches"])
    season_matches["season"] = season
    season_tables.append(season_matches)

matches = pd.concat(season_tables, ignore_index=True)

matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values("date").reset_index(drop=True)
# Use the alternative score column wherever score.ft is missing.
matches["score.ft"] = matches["score.ft"].fillna(matches["score"])

### Three Season Matches

In [3]:
original_season_tables = []

for season in ["2023_24", "2024_25", "2025_26"]:
    filename = f"premier_league_{season}.json"

    with open(filename, encoding="utf-8") as file:
        data = json.load(file)

    season_matches = pd.json_normalize(data["matches"])
    season_matches["season"] = season
    original_season_tables.append(season_matches)

original_matches = pd.concat(original_season_tables, ignore_index=True)

original_matches["date"] = pd.to_datetime(original_matches["date"])
original_matches = original_matches.sort_values("date").reset_index(drop=True)
# Use the alternative score column wherever score.ft is missing.
original_matches["score.ft"] = original_matches["score.ft"].fillna(original_matches["score"])

In [4]:
validation_fixtures = []

for train_indices,valid_indices in TimeSeriesSplit(n_splits=3).split(original_matches):
    validation_fixtures.append(
        original_matches.iloc[valid_indices][['season', 'team1', 'team2']]
        )

In [5]:
matches["home_goals"] = matches["score.ft"].str[0]
matches["away_goals"] = matches["score.ft"].str[1]

matches[
    ["date", "team1", "team2", "home_goals", "away_goals"]
].head()

,date,team1,team2,home_goals,away_goals
0,2021-08-13,Brentford FC,Arsenal FC,2,0
1,2021-08-14,Manchester United FC,Leeds United FC,5,1
2,2021-08-14,Watford FC,Aston Villa FC,3,2
3,2021-08-14,Leicester City FC,Wolverhampton Wanderers FC,1,0
4,2021-08-14,Everton FC,Southampton FC,3,1


In [6]:
# Confirm every match has both goal totals.
assert matches[["home_goals", "away_goals"]].notna().all().all()

# Collect all team names across the three seasons.
teams = pd.concat([matches["team1"], matches["team2"]]).unique()

In [7]:
def expected_result(home_rating, away_rating, home_advantage=0):
    # returns the expected score for the home team given their rating and the away team's rating.
    return 1 / (1 + (10 ** ((away_rating - (home_rating + home_advantage)) / 400)))

In [8]:
def update_ratings(home_rating, away_rating, home_goals, away_goals, k=40, home_advantage=0):
    # Return the updated home and away ratings.
    exp_result = expected_result(home_rating, away_rating, home_advantage)
    if home_goals > away_goals:
        act_result = 1
    elif home_goals == away_goals:
        act_result = 0.5
    else:
        act_result = 0
    change = k * (act_result - exp_result)
    home_rating = home_rating + change
    away_rating = away_rating - change
    return (home_rating, away_rating)

In [9]:
# Give every team an initial rating.
ratings = {team: 1500.0 for team in teams}

home_elo_before = []
away_elo_before = []

for match in matches.itertuples(index=False):
    # Update the ratings using the result of a match
    home_team = match.team1
    away_team = match.team2
    home_elo_before.append(ratings[home_team])
    away_elo_before.append(ratings[away_team])
    ratings[home_team], ratings[away_team] = update_ratings(home_elo_before[-1], 
                                        away_elo_before[-1], 
                                        match.home_goals,
                                        match.away_goals,
                                        home_advantage=50)



In [10]:
# Create new prematch elo columns within the matches DataFrame
matches['home_elo_before'] = home_elo_before
matches['away_elo_before'] = away_elo_before

# Create new prematch predictions based on which team has more elo
matches['elo_difference'] = matches['home_elo_before'] - matches['away_elo_before']

In [11]:
def match_result(home_goals, away_goals):
    # Return a categorical value for the result of a match using scoreline
    if home_goals > away_goals:
        return 0
    elif home_goals == away_goals:
        return 1
    elif home_goals < away_goals:
        return 2

In [12]:
# Create match result column for model predictions 
matches['result'] = (matches[['home_goals', 'away_goals']]
                     .apply(lambda x: match_result(x.home_goals, 
                                                   x.away_goals), 
                                                   axis=1))

In [13]:
# Validates new columns to ensure all values are present
print(matches[['home_elo_before', 'away_elo_before', 'elo_difference', 'result']].isna().sum())

# Validates that each fixture per season only occured once (accounting for home/away)
print(matches.duplicated(subset=['season', 'team1', 'team2']).sum())

# Ensures that the date column never decreases to ensure matches are chronological
print(matches['date'].is_monotonic_increasing)

home_elo_before    0
away_elo_before    0
elo_difference     0
result             0
dtype: int64
0
True


In [14]:
# Creates a dictionary with a list containing the points won from each match for every team in chronological order
match_history = {team:[] for team in teams}

def points_won(home_goals, away_goals):
    # Returns the number of points won from a match
    if home_goals > away_goals:
        return (3, 0)
    elif home_goals == away_goals:
        return (1, 1)
    elif home_goals < away_goals:
        return (0, 3)

home_pre_match_average = []
away_pre_match_average = []

for match in matches.itertuples(index=False):
    # Records the points won for every team in every match in chronological order
    # Keeps a running point average of the last five games played by each team
    home_team = match.team1
    away_team = match.team2
    if match_history[home_team]:
        home_pre_match_average.append(np.mean(match_history[home_team][-5:]))
    else:
        home_pre_match_average.append(np.nan)
    if match_history[away_team]:
        away_pre_match_average.append(np.mean(match_history[away_team][-5:]))
    else:
        away_pre_match_average.append(np.nan)
    match_history[home_team].append(points_won(match.home_goals, match.away_goals)[0])
    match_history[away_team].append(points_won(match.home_goals, match.away_goals)[1])

# Check to ensure that column lengths match before adding to the matches DataFrame
print(len(home_pre_match_average) == len(matches))
print(len(away_pre_match_average) == len(matches))

matches['home_form_5'] = home_pre_match_average
matches['away_form_5'] = away_pre_match_average

matches.loc[
    (matches['team1'] == 'Manchester City FC') |
    (matches['team2'] == 'Manchester City FC'),
    ['date', 'team1', 'team2', 'home_form_5', 'away_form_5']
].head()

True
True


,date,team1,team2,home_form_5,away_form_5
9,2021-08-15,Tottenham Hotspur FC,Manchester City FC,NaN,NaN
13,2021-08-21,Manchester City FC,Norwich City FC,0.00,0.00
21,2021-08-28,Manchester City FC,Arsenal FC,1.50,0.00
34,2021-09-11,Leicester City FC,Manchester City FC,2.00,2.00
41,2021-09-18,Manchester City FC,Southampton FC,2.25,0.75


In [15]:
# Creates two dictionaries with lists containing the goals scored and conceeded from each match for every team in chronological order
goal_scored_history = {team:[] for team in teams}
goal_conceded_history = {team:[] for team in teams}


home_scored_average = []
home_conceded_average = []

away_scored_average = []
away_conceded_average = []


for match in matches.itertuples(index=False):
    # Records the goals scored and conceeded for every team in every match in chronological order
    # Keeps a running pre-match goals average of the last five games played by each team
    home_team = match.team1
    away_team = match.team2
    if goal_scored_history[home_team]:
        home_scored_average.append(np.mean(goal_scored_history[home_team][-5:]))
        home_conceded_average.append(np.mean(goal_conceded_history[home_team][-5:]))
    else:
        home_scored_average.append(np.nan)
        home_conceded_average.append(np.nan)
    if goal_conceded_history[away_team]:
        away_scored_average.append(np.mean(goal_scored_history[away_team][-5:]))
        away_conceded_average.append(np.mean(goal_conceded_history[away_team][-5:]))
    else:
        away_scored_average.append(np.nan)
        away_conceded_average.append(np.nan)
    goal_scored_history[home_team].append(match.home_goals)
    goal_conceded_history[home_team].append(match.away_goals)
    goal_scored_history[away_team].append(match.away_goals)
    goal_conceded_history[away_team].append(match.home_goals)

matches['home_goals_scored_5'] = home_scored_average
matches['home_goals_conceded_5'] = home_conceded_average
matches['away_goals_scored_5'] = away_scored_average
matches['away_goals_conceded_5'] = away_conceded_average

matches.loc[
    (matches['team1'] == 'Manchester City FC') |
    (matches['team2'] == 'Manchester City FC'),
    ['date', 'team1', 'team2', 'home_goals_scored_5', 'home_goals_conceded_5', 'away_goals_scored_5', 'away_goals_conceded_5']
].head()

,date,team1,team2,home_goals_scored_5,home_goals_conceded_5,away_goals_scored_5,away_goals_conceded_5
9,2021-08-15,Tottenham Hotspur FC,Manchester City FC,NaN,NaN,NaN,NaN
13,2021-08-21,Manchester City FC,Norwich City FC,0.000000,1.000000,0.000000,3.000000
21,2021-08-28,Manchester City FC,Arsenal FC,2.500000,0.500000,0.000000,2.000000
34,2021-09-11,Leicester City FC,Manchester City FC,1.333333,1.666667,3.333333,0.333333
41,2021-09-18,Manchester City FC,Southampton FC,2.750000,0.250000,1.000000,1.500000


### XGBoost Model Training

In [16]:
# Create a list of columns that we want our model to use to predict
input_columns = ['home_elo_before', 'away_elo_before', 'elo_difference', 'home_form_5', 'away_form_5', 'home_goals_scored_5', 'home_goals_conceded_5', 'away_goals_scored_5', 'away_goals_conceded_5']

# Create our X and y variables, which contain the columns we predict with and predict for respectively
X = matches[input_columns]
y = matches['result']

In [17]:
# Identify each match using its season, home team, and away team.
fixture_columns = ['season', 'team1', 'team2']
match_ids = pd.MultiIndex.from_frame(matches[fixture_columns])

cv_splits = []

for fixtures in validation_fixtures:
    fixture_ids = pd.MultiIndex.from_frame(fixtures[fixture_columns])

    # Find the positions of the original validation fixtures in matches.
    valid_mask = match_ids.isin(fixture_ids)
    valid_indices = np.flatnonzero(valid_mask)

    # Train only on matches before the first validation date.
    valid_date = matches['date'].iloc[valid_indices].min()
    train_mask = matches['date'] < valid_date
    train_indices = np.flatnonzero(train_mask)

    cv_splits.append((train_indices, valid_indices))

# Check fold sizes and chronological separation.
for train_indices, valid_indices in cv_splits:
    print("Training matches:", len(train_indices))
    print("Validation matches:", len(valid_indices))
    print("Last training date:", matches['date'].iloc[train_indices].max())
    print("First validation date:", matches['date'].iloc[valid_indices].min())
    print()

Training matches: 1043
Validation matches: 285
Last training date: 2024-03-17 00:00:00
First validation date: 2024-03-30 00:00:00

Training matches: 1329
Validation matches: 285
Last training date: 2025-01-01 00:00:00
First validation date: 2025-01-04 00:00:00

Training matches: 1610
Validation matches: 285
Last training date: 2025-10-26 00:00:00
First validation date: 2025-11-01 00:00:00



In [18]:
# Creates a dummy baseline to compare the XGBoost Classifier to
baseline = DummyClassifier(strategy='prior')
baseline_results = cross_validate(baseline, X, y, 
                                  cv=cv_splits, 
                                  scoring='neg_log_loss'
                                  )
baseline_score = -1 * baseline_results['test_score'].mean()
print(baseline_score)

1.0815141131447572


### Baseline Model Results

- Three Season Model log loss: 1.0828
- Five Season Model log loss: 1.0815

In [19]:
# Creates a dictionary of hyperparameter values for the model to tune to
param_grid = {
    'n_estimators':[100, 300],
    'learning_rate':[0.03, 0.1],
    'max_depth':[2, 3, 4],
    'min_child_weight':[1, 5]
}

# Creates a model for every hyperparameter configuration and returns best cross-validated score
search = GridSearchCV(
    estimator = XGBClassifier(n_jobs=4, random_state=0),
    param_grid=param_grid,
    scoring='neg_log_loss',
    cv=cv_splits,
    n_jobs=1,
    verbose=1,
    refit=True
)

# Fit the tuned model with the data
search.fit(X, y)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'learning_rate': [0.03, 0.1], 'max_depth': [2, 3, ...], 'min_child_weight': [1, 5], 'n_estimators': [100, 300]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_log_loss'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.","[(array([ 0, ...shape=(1043,)), ...), (array([ 0, ...shape=(1329,)), ...), ...]"
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` param

In [20]:
# Display results from the grid search
print(search.best_params_)
print(-search.best_score_)
print(search.best_estimator_)

print(-search.cv_results_['split0_test_score'][search.best_index_])
print(-search.cv_results_['split1_test_score'][search.best_index_])
print(-search.cv_results_['split2_test_score'][search.best_index_])

{'learning_rate': 0.03, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 100}
1.0104255676269531
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.03, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=2,
              max_leaves=None, min_child_weight=1, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=4, num_parallel_tree=None, ...)
0.9950481057167053
0.9904801249504089
1.0457484722137451


### Initial model results

- Features: home Elo, away Elo, and Elo difference.
- Validation: three chronological folds.
- Baseline log loss: 1.0828.
- XGBoost log loss: 1.0321.

XGBoost reduced validation log loss by approximately 4.7%.

- Added points-form columns which are updated with the average points won by each home and away team to keep track of current form
- XGBoost log loss: 1.0300
- Change is minor, haven't established if it can consistently improve future predictions

XGBoost reduced validation log loss by approximately 0.203%

- Added goals-scored and goals-conceded columns for both home and away teams, which keep track of the average goals scored/conceded over the last five matches by each team leading up to the fixture
- XGBoost log loss: 1.0299
- Change is yet again minor, time to either import new data for better insights or more on to other methods of predictionj

XGBoost reduced validation log loss by approximately 0.004%

- Added home-field advantage for the home team when calculating elo and expected match result
- XGBoost log loss: 1.0316

XGBoost increased validation log loss by approximately 0.158%

- Tested K values of 10, 20, and 40 to compare slower and faster Elo adaptation around the initial setting of 20.
- Most optimal k value out of the three was 20 (No Change)
- XGBoost log loss: 1.0316

XGBoost had no change on validation log loss

- Updated training data to include five seasons of previous data rather than three seasons
- XGBoost log loss: 1.0104

XGBoost reduced validation log loss by approxmately 2.09%

### Logistic Regression Comparison

In [21]:
# Create a Pipeline to prepare the data and build the model to compare to XGBClassifier
logistic_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

logistic_results = cross_validate(logistic_pipeline, X, y, 
                                  cv=cv_splits, 
                                  scoring='neg_log_loss', 
                                  return_estimator=True)
logistic_score = -1 * logistic_results['test_score'].mean()

print(logistic_score)
print(-logistic_results['test_score'])

0.9991213254703544
[0.97864793 0.97662527 1.04209078]


### Probability Calibration Checks

Collected home-win, draw, and away-win probabilities for 855 validation
matches using the logistic regression pipeline fitted for each
chronological fold. Each model was trained only on earlier matches.

For each outcome, grouped predicted probabilities into five intervals
of width 0.2 and compared:
- Mean predicted probability.
- Actual outcome frequency.
- Number of matches.

Also examined home-win calibration separately by fold to check whether
the observed discrepancies persisted across validation periods.

These checks diagnose probability quality; they do not modify the model.

In [22]:
# Collect probabilities and actual outcomes for each model's validation fold.
# Combine folds in matching order for calibration checks.
# Probability columns: 0 = home win, 1 = draw, 2 = away win.

validation_probabilities = []
validation_labels = []

for model, (_, valid_indices) in zip(
    logistic_results['estimator'], cv_splits
):
    validation_probabilities.append(
        model.predict_proba(X.iloc[valid_indices])
    )
    validation_labels.append(y.iloc[valid_indices].to_numpy())

validation_probabilities = np.concatenate(validation_probabilities)
validation_labels = np.concatenate(validation_labels)

print(validation_probabilities.shape)
print(validation_labels.shape)
print(logistic_results['estimator'][0].classes_)

(855, 3)
(855,)
[0 1 2]


In [23]:
# Check overall home-win calibration by comparing average predicted probability with the actual home-win rate

home_win_check = pd.DataFrame({
    'predicted_probability': validation_probabilities[:, 0],
    'actual_home_win': validation_labels == 0
})

home_win_check['probability_bin'] = pd.cut(
    home_win_check['predicted_probability'],
    bins=np.linspace(0, 1, 6),
    include_lowest=True
)

home_calibration = home_win_check.groupby(
    'probability_bin',
    observed=True
).agg(
    mean_predicted=('predicted_probability', 'mean'),
    actual_win_rate=('actual_home_win', 'mean'),
    match_count=('actual_home_win', 'size')
)

print(home_calibration)

                 mean_predicted  actual_win_rate  match_count
probability_bin                                              
(-0.001, 0.2]          0.135006         0.157303           89
(0.2, 0.4]             0.306043         0.260870          253
(0.4, 0.6]             0.498052         0.484211          285
(0.6, 0.8]             0.686653         0.590674          193
(0.8, 1.0]             0.844110         0.800000           35


In [24]:
# Check overall draw calibration by comparing average predicted probability with the actual draw rate


draw_check = pd.DataFrame({
    'predicted_probability': validation_probabilities[:, 1],
    'actual_draw': validation_labels == 1
})

draw_check['probability_bin'] = pd.cut(
    draw_check['predicted_probability'],
    bins=np.linspace(0, 1, 6),
    include_lowest=True
)

draw_calibration = draw_check.groupby(
    'probability_bin',
    observed=True
).agg(
    mean_predicted=('predicted_probability', 'mean'),
    actual_draw_rate=('actual_draw', 'mean'),
    match_count=('actual_draw', 'size')
)

print(draw_calibration)

                 mean_predicted  actual_draw_rate  match_count
probability_bin                                               
(-0.001, 0.2]          0.167618          0.233831          201
(0.2, 0.4]             0.243393          0.264526          654


In [25]:
# Check overall away-win calibration by comparing average predicted probability with the actual away_win rate


away_win_check = pd.DataFrame({
    'predicted_probability': validation_probabilities[:, 2],
    'actual_away_win': validation_labels == 2
})

away_win_check['probability_bin'] = pd.cut(
    away_win_check['predicted_probability'],
    bins=np.linspace(0, 1, 6),
    include_lowest=True
)

away_win_calibration = away_win_check.groupby(
    'probability_bin',
    observed=True
).agg(
    mean_predicted=('predicted_probability', 'mean'),
    actual_away_win_rate=('actual_away_win', 'mean'),
    match_count=('actual_away_win', 'size')
)

print(away_win_calibration)

                 mean_predicted  actual_away_win_rate  match_count
probability_bin                                                   
(-0.001, 0.2]          0.115508              0.148936          282
(0.2, 0.4]             0.292343              0.307692          299
(0.4, 0.6]             0.485365              0.457286          199
(0.6, 0.8]             0.680087              0.661765           68
(0.8, 1.0]             0.836491              0.714286            7


In [26]:
home_win_check['fold'] = np.concatenate([
    np.full(len(valid_indices), fold_number)
    for fold_number, (_, valid_indices) in enumerate(cv_splits, start=1)
])

home_calibration_by_fold = home_win_check.groupby(
    ['fold', 'probability_bin'],
    observed=True
).agg(
    mean_predicted=('predicted_probability', 'mean'),
    actual_win_rate=('actual_home_win', 'mean'),
    match_count=('actual_home_win', 'size')
)

home_calibration_by_fold

mean_predicted  actual_win_rate  match_count
fold probability_bin                                              
1    (-0.001, 0.2]          0.135831         0.178571           28
     (0.2, 0.4]             0.306146         0.273684           95
     (0.4, 0.6]             0.496088         0.468354           79
     (0.6, 0.8]             0.684858         0.587302           63
     (0.8, 1.0]             0.852530         0.750000           20
2    (-0.001, 0.2]          0.127633         0.151515           33
     (0.2, 0.4]             0.306336         0.270270           74
     (0.4, 0.6]             0.502133         0.500000          104
     (0.6, 0.8]             0.698243         0.629032           62
     (0.8, 1.0]             0.831248         0.833333           12
3    (-0.001, 0.2]          0.142870         0.142857           28
     (0.2, 0.4]             0.305670         0.238095           84
     (0.4, 0.6]             0.495412         0.480392          102
     (0.6, 0.8]             0.677750         0.558824           68
     (0.8, 1.0]             0.839417         1.000000            3

### Calibration Findings

- Home wins: predictions in the 60–80% range averaged 68.7%, while
  the actual home-win rate was 59.1% across 193 matches.
  Overestimation appeared in all three folds.
- Draws: predicted probabilities underestimated observed draw rates
  in both populated bins: 16.8% versus 23.4%, and 24.3% versus 26.5%.
- Away wins: predicted and observed rates were relatively close in
  the larger bins. The highest-probability bin contained only seven
  matches, limiting interpretation.

These are descriptive validation findings, subject to sampling variation
and bin choice. No calibration adjustment has been applied.
A potential next experiment is to fit an adjustment using earlier data
and evaluate it on later matches using log loss.

### Logistic Regression Model Results

- Built a Pipeline to pre-process the data, as linear regression models cannot handle empty entries
- Fit and validated results using X and y
- Logistic Regression log loss: 1.0116

Logistic Regression reduced validation log loss by approximately 1.79% compared to XGBoost  
Logistic Regression reduced validation log loss by approximately 6.58% compared to Baseline

- Added home-field advantage when calculating elo and expected match result
- Logistic Regression log loss: 1.0114

Logistic Regression reduced validation log loss by 0.015%

- Tested K values of 10, 20, and 40 to compare slower and faster Elo adaptation around the initial setting of 20.
- Most optimal k value out of the three was 40 (attempted 80 to see if 40 wasn't the limit, made model worse)
- Logistic Regression log loss: 1.0068

Logistic Regression reduced validated log loss by 0.456%

- Updated training data to include five seasons of previous data rather than three seasons
- Logistic Regression log loss: 0.99912

Logistic Regression reduced validated log loss by 0.761%



### Calibration Adjustment Experiment

In [41]:
calibration_results = []

for fold_number, (train_indices, valid_indices) in enumerate(cv_splits, start=1):
    # Divide this fold's training data chronologically.
    cut_position = int(len(train_indices) * 0.8)
    calibration_start_date = matches['date'].iloc[train_indices[cut_position]]
    training_dates = matches['date'].iloc[train_indices]

    fit_indices = train_indices[training_dates < calibration_start_date]
    calibration_indices = train_indices[training_dates >= calibration_start_date]

    # Train a fresh base pipeline.
    base_model = clone(logistic_pipeline)
    base_model.fit(X.iloc[fit_indices], y.iloc[fit_indices])

    # Learn the adjustment from separate calibration matches.
    calibration_probs = base_model.predict_proba(X.iloc[calibration_indices])
    calibration_inputs = np.log(np.clip(calibration_probs, 1e-10, 1))

    calibrator = LogisticRegression(max_iter=1000)
    calibrator.fit(calibration_inputs, y.iloc[calibration_indices])

    # Evaluate both versions on the same validation matches.
    original_probs = base_model.predict_proba(X.iloc[valid_indices])
    adjusted_probs = calibrator.predict_proba(
        np.log(np.clip(original_probs, 1e-10, 1))
    )

    calibration_results.append({
        'fold': fold_number,
        'uncalibrated': log_loss(
            y.iloc[valid_indices], original_probs, labels=[0, 1, 2]
        ),
        'calibrated': log_loss(
            y.iloc[valid_indices], adjusted_probs, labels=[0, 1, 2]
        ),
        'full_training': -logistic_results['test_score'][fold_number - 1]
    })

calibration_results = pd.DataFrame(calibration_results)

print(calibration_results)
print("\nMean log losses:")
print(calibration_results[
    ['uncalibrated', 'calibrated', 'full_training']
].mean())

   fold  uncalibrated  calibrated  full_training
0     1      0.981673    0.973993       0.978648
1     2      0.982412    0.979755       0.976625
2     3      1.047955    1.037624       1.042091

Mean log losses:
uncalibrated     1.004014
calibrated       0.997124
full_training    0.999121
dtype: float64
